## 1. Background

Maternal mortality and pregnancy-related complications remain a critical global health concern. Many adverse outcomes — pre-eclampsia, gestational diabetes, hypertensive disorders, and cardiovascular distress — can be anticipated and mitigated if **high-risk pregnancies are identified early** using routinely collected clinical indicators.

The **Maternal Health and High-Risk Pregnancy** dataset contains patient records combining maternal age, vital signs (blood pressure, heart rate, body temperature), a metabolic marker (blood sugar), body composition (BMI), medical/psychological history flags (previous complications, pre-existing diabetes, gestational diabetes, mental health status), and a clinician-assigned outcome label — **Risk Level** (`High` / `Low`).

Because a labeled outcome already exists in the data, this is framed as a **supervised classification problem**. This document defines the business context, objectives, constraints, and success criteria that will guide the entire CRISP-DM process — **before** any statistical exploration of the data itself, which belongs to the next phase (Data Understanding).

---

## 2. Business Objectives

| # | Objective |
|---|-----------|
| 1 | Enable early identification of **high-risk pregnancies** based on measurable clinical indicators. |
| 2 | Support healthcare providers and maternal health programs with a **data-driven risk-screening tool** to prioritize monitoring and interventions. |
| 3 | Reduce preventable maternal and fetal complications by flagging known risk factors (e.g., abnormal blood sugar, blood pressure, diabetes, obesity) before they escalate. |
| 4 | Provide clinicians with an **interpretable and trustworthy** decision-support reference rather than a black-box output. |
| 5 | Lay the groundwork for a decision-support system that could be integrated into antenatal care workflows or telehealth triage systems. |

### Why this matters
Identifying high-risk cases early allows clinicians to allocate resources (specialist referrals, more frequent monitoring, lifestyle interventions) more effectively. Established obstetric knowledge already links factors such as hypertension, diabetes, extreme BMI, advanced maternal age, and prior complications to elevated pregnancy risk — this project aims to confirm and quantify these relationships using real patient data in the next phase, and eventually operationalize them into a screening tool.

---

## 3. Business Success Criteria

The project will be considered successful from a business perspective if:

- A clear, interpretable set of **risk indicators** is identified that aligns with recognized clinical knowledge about pregnancy risk.
- The resulting analysis/model can classify a patient as **High** or **Low** risk with sufficient reliability to be useful as a **screening aid** (not a diagnostic replacement).
- The findings are **explainable** to non-technical stakeholders (clinicians, health-program managers) in plain clinical language.
- The approach is **generalizable** enough to be applied to new, unseen patient records collected through routine antenatal visits.

---

## 4. Situation Assessment

### 4.1 Inventory of Resources
- **Data**: `Pregnancy.csv`, containing maternal vital signs, lab-like measurements, medical/psychological history flags, and a clinician-assigned risk label.
- **Tools**: Python (pandas, scikit-learn, matplotlib/seaborn) for statistical analysis and modeling in later phases.
- **Domain knowledge**: General maternal-health clinical reference ranges (blood pressure, blood sugar, BMI categories) that will be used to interpret findings once the data is explored.

### 4.2 Requirements, Assumptions, and Constraints
- **Requirement**: The target variable (`Risk Level`) must be reliable and consistently labeled, as it anchors all subsequent modeling.
- **Assumption**: The dataset is reasonably representative of a real patient population.
- **Constraint**: This is an **observational, retrospective dataset** — any model built is a decision-support/screening tool, **not a diagnostic device**, and should not replace clinical judgment.
- **Constraint**: The dataset size is limited (on the order of a few thousand records or fewer), which favors simpler, more interpretable models and cross-validation over highly complex approaches.
- **Constraint**: Data quality (missing values, potential data-entry errors, class balance) is unknown at this stage and must be formally assessed in Data Understanding before any conclusions are drawn.

### Prediction-Time Availability Rule

A feature is usable in the deployable model only if it would be available **before** the risk decision is made. This rule is the basis for separating a **leakage-excluded dataset** (used for the final deployable model) from a **leakage-inclusive comparison dataset** (used only to quantify the effect of leakage). Features that are determined during the same pregnancy episode, after the outcome is assigned, or through post-diagnosis reasoning are treated as leakage-prone and must be excluded from the final deployable model.

- **Dataset facts**: the observed values in `Pregnancy.csv` include maternal age, vital signs, blood sugar, BMI, medical history flags, and the physician-assigned `Risk Level` label.
- **Project/modeling assumptions**: the model is intended as an early prenatal screening support tool; missing values will be handled with training-only statistics; the final test set will be held out until the final evaluation stage.
- **Clinical interpretation**: observed relationships in the data describe associations and do not establish causality or clinical necessity.

### Leakage-Prone Features and Timing

`Gestational Diabetes` is explicitly treated as leakage-prone for the deployable model, because it is diagnosed during the same pregnancy and is therefore not available at the point an early risk decision would be made. The same reasoning applies to engineered features such as BMI-derived risk categories and other variables that depend on a diagnosis or on the same clinical process that generated the target. These variables may be retained only in a **leakage-inclusive comparison** to quantify the effect of leakage — not as part of the final deployable model.

### 4.3 Risks and Contingencies
| Risk | Contingency |
|------|-------------|
| Missing or incomplete data reduces reliability | Assess thoroughly in Data Understanding; impute or drop records in Data Preparation |
| Implausible or erroneous values distort statistics | Identify in Data Understanding; clean in Data Preparation |
| Model overfits due to limited sample size | Use cross-validation and simpler, interpretable models |
| Class imbalance skews predictions toward the majority class | Assess balance in Data Understanding; apply class weighting or resampling if needed |
| Findings misinterpreted as diagnostic rather than screening tools | Clearly document limitations and intended use in final deliverables |
| Some features may correlate almost perfectly with the target (potential target leakage) | Investigate explicitly in Data Understanding and Modeling before trusting such features |

### 4.4 Terminology
- **High-Risk Pregnancy**: A pregnancy with elevated likelihood of complications for mother and/or fetus.
- **Gestational Diabetes**: Diabetes diagnosed during pregnancy, distinct from pre-existing diabetes.
- **BS (Blood Sugar)**: Blood glucose level, a key indicator of diabetes-related risk.
- **BMI (Body Mass Index)**: Weight-to-height ratio; standard categories: <18.5 underweight, 18.5–24.9 normal, 25–29.9 overweight, 30+ obese.
- **Systolic/Diastolic BP**: The two blood-pressure readings; elevated values indicate hypertension, a major pregnancy-risk factor (e.g., pre-eclampsia).

---

## 5. Data Field Descriptions (Business / Clinical Meaning)

Before any statistical exploration (which belongs to Data Understanding), it is important to establish what each field in the dataset actually means from a clinical and business standpoint, and why it was likely collected. This shared understanding is what allows domain knowledge and data analysis to be connected later.

| # | Field | Type | Clinical / Business Meaning |
|---|---|---|---|
| 1 | **Age** | Numeric (years) | The mother's age at the time of the recorded pregnancy. Clinically relevant because both very young and advanced maternal age are established risk categories in obstetrics. |
| 2 | **Systolic BP** | Numeric (mmHg) | The higher of the two blood-pressure readings, measured when the heart contracts. Elevated systolic pressure is a core indicator of hypertensive disorders of pregnancy, including gestational hypertension and pre-eclampsia. |
| 3 | **Diastolic** | Numeric (mmHg) | The lower blood-pressure reading, measured when the heart relaxes between beats. Used alongside systolic pressure to assess hypertension; diastolic elevation is often considered even more specific to pre-eclampsia risk. |
| 4 | **BS (Blood Sugar)** | Numeric | Blood glucose level. A central metabolic marker used to assess diabetes-related risk, including both pre-existing and gestational diabetes; elevated levels are linked to macrosomia, birth complications, and long-term metabolic risk for both mother and child. |
| 5 | **Body Temp** | Numeric (°F) | The mother's body temperature. Elevated readings may indicate infection or fever, both of which are recognized contributors to pregnancy complications. |
| 6 | **BMI** | Numeric | Body Mass Index — a ratio of weight to height used to classify body composition (underweight, normal, overweight, obese). Both extremes are associated with elevated pregnancy risk (e.g., nutritional deficiency at low BMI; hypertension and gestational diabetes at high BMI). |
| 7 | **Previous Complications** | Binary (0/1) | Indicates whether the patient has a documented history of complications in a prior pregnancy. A strong clinical predictor, since obstetric history is one of the most reliable indicators of recurring risk. |
| 8 | **Preexisting Diabetes** | Binary (0/1) | Indicates whether the patient was diagnosed with diabetes *before* becoming pregnant. This is distinct from gestational diabetes and generally carries a well-established, elevated risk profile. |
| 9 | **Gestational Diabetes** | Binary (0/1) | Indicates whether the patient developed diabetes *during* the current pregnancy. A recognized major risk factor, though — because it is diagnosed as part of the same clinical workup that determines risk — its relationship with the target should be interpreted carefully (see Section 4.3, Risks and Contingencies). |
| 10 | **Mental Health** | Binary (0/1) | Indicates a flag for a mental health concern (e.g., stress, anxiety, depression) noted during antenatal care. Increasingly recognized in obstetric practice as a contributor to overall pregnancy risk, alongside physical indicators. |
| 11 | **Heart Rate** | Numeric (bpm) | The mother's resting heart rate. Used as a general indicator of cardiovascular and physiological stress; abnormal values can reflect anemia, infection, or cardiac strain. |
| 12 | **Risk Level** (target) | Categorical (`High` / `Low`) | The clinician-assigned outcome label representing the overall pregnancy risk classification. This is the variable the project aims to predict. |

### Notes on Field Interpretation
- All numeric fields above are expected to fall within physiologically plausible ranges for a pregnant population; verifying this is a task for Data Understanding, not this document.
- The four binary flags (Previous Complications, Preexisting Diabetes, Gestational Diabetes, Mental Health) function as a compact medical history summary rather than real-time vitals, and are expected — from clinical knowledge alone — to carry significant predictive weight.
- The units and exact measurement protocol behind each field (e.g., whether BP was measured in a clinical setting, whether BS reflects fasting or non-fasting glucose) are not specified in the raw data and should be clarified or noted as a limitation.

---

## 6. Data Mining Goals

Translating the business objectives above into technical, data-mining terms:

1. Build a **binary classification model** predicting `Risk Level` (High / Low) from the available clinical features.
2. Empirically determine, through the Data Understanding phase, **which features are most strongly associated with high-risk status**, and validate these against clinical expectations.
3. Identify whether any features show a **suspiciously deterministic relationship** with the target (potential target leakage) that would need special handling.
4. Detect and resolve **data quality issues** (missing values, implausible values, duplicates) before modeling.
5. Evaluate whether a **simple, interpretable rule-based screening approach** could complement or compete with more complex models — important for clinical trust and adoption.

---

## 7. Data Mining Success Criteria

- A classification model with strong **recall on the High-risk class** (missing a truly high-risk patient is far more costly than a false alarm).
- Feature importance rankings that are **clinically plausible** and can be cross-checked against established obstetric risk knowledge.
- A final model or rule-set that can be explained in plain language to a clinician.

---

## 8. Preliminary Project Plan (High-Level)

| CRISP-DM Phase | Planned Focus |
|---|---|
| **Business Understanding** | *(this document)* Objectives, constraints, and success criteria |
| **Data Understanding** | Data description, quality assessment, exploratory analysis, correlations, feature-vs-target relationships, threshold effects, and leakage investigation |
| **Data Preparation** | Clean data, impute missing values, engineer threshold-based features, encode categorical target, address class imbalance, split train/test |
| **Modeling** | Train and compare classifiers (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting) and a simple rule-based baseline |
| **Evaluation** | Assess accuracy, precision, recall, F1, ROC-AUC; validate that feature importances align with domain knowledge; check for leakage effects |
| **Deployment** | Summarize into a report/dashboard or an illustrative risk-scoring tool, with a clear "screening aid, not diagnosis" disclaimer |

---

## 9. Questions This Project Aims to Answer

The subsequent phases of this project (starting with Data Understanding) are designed to answer:

1. What clinical factors are most strongly associated with high-risk pregnancy in this dataset?
2. Does maternal age correlate with pregnancy risk, and is there a critical age range?
3. How do blood pressure levels differ between risk groups, and is there a clinical cutoff?
4. Is elevated blood sugar a strong predictor, and at what level does risk change?
5. Does BMI increase pregnancy risk, and in which direction (under/overweight)?
6. Do patients with previous pregnancy complications face higher risk again?
7. Is there a relationship between mental health status and pregnancy risk?
8. Do heart rate and body temperature differ meaningfully between risk groups?
9. Can a model reliably predict risk level, and which features matter most?
10. What combinations of risk factors most reliably signal high-risk pregnancy?
11. Are there signs of label leakage or overly deterministic rules in the target variable?

> Note: These are the **guiding questions** for the project. Their answers, backed by actual computed statistics, belong to the Data Understanding phase — not this document — since Business Understanding is concerned with defining the problem and goals, not analyzing the data itself.

---

## 10. Clinical Decision Framing

This section translates clinical priorities into explicit design decisions that will guide Data Mining Success Criteria (Section 7) and later the Evaluation phase. These are framing questions answered from business/clinical reasoning — not from computed model results, which belong to Evaluation.

### Question A — Relative Cost of Errors
**What is the relative cost of a False Negative versus a False Positive? Which error should the model minimize more aggressively, and why?**

A false negative can miss a genuinely high-risk pregnancy and delay referral or intervention, so its clinical cost is substantially higher than the inconvenience and resource use of a false positive. The screening model should therefore minimize false negatives more aggressively, while still monitoring false positives to avoid unnecessary anxiety and workload. This is why Section 7 (Data Mining Success Criteria) prioritizes **Recall on the High-risk class** over overall accuracy.

### Question B — Minimum Acceptable Recall
**What is the minimum acceptable Recall for the High-risk class for this tool to be usable in clinical practice?**

A minimum High-risk Recall target of **95%** is set here as the project's design objective for an initial screening tool, reflecting the clinical priority of detecting nearly all high-risk pregnancies. This is a **target to design and evaluate against**, not a result — the actual Recall achieved by any trained model is an empirical outcome that belongs to the Modeling/Evaluation phases and must be reported and compared against this 95% objective there, not assumed or stated here.

### Question C — Point of Clinical Application
**At which stage of prenatal care will this model realistically be applied, and how does that affect the features used?**

The intended use is the **first prenatal visit** or an early routine screening assessment, when baseline history, vital signs, BMI, blood sugar, and mental-health information are typically available. Features that are only known after later testing, or only after the risk label itself is determined, must be excluded or explicitly time-stamped to prevent target leakage — directly motivating the Prediction-Time Availability Rule in Section 4.2.

### Question D — Feature-by-Feature Timing Check
**For each modeling feature, is it available when risk is assessed, or could it be part of the process that determines Risk Level?**

- Age, Systolic BP, Diastolic BP, BS, Body Temperature, BMI, Previous Complications, Preexisting Diabetes, Mental Health, and Heart Rate are generally available at intake, if collected as part of a standard first-visit assessment.
- **Gestational Diabetes** requires special timing justification: if it is diagnosed during the same care process, or encoded from the final risk assessment itself, it constitutes target leakage and must be excluded from an early-screening model. It is usable only when it can be documented as known **before** the prediction point.